In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost.callback import EarlyStopping
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,RobustScaler,OneHotEncoder
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
#load dataset
df = pd.read_csv("Final_dataset_83.csv")
print(df.shape)
print(xgb.__version__)
print(df.columns)

X_features =['critical_temp', 'subfamily', 'family', 'low_confidence', 'material',
       'is_ambiguous', 'vF_predicted_log10',
       'prediction_source']
numeric_features     = [c for c in df.columns if c != X_features] 
categorical_features = ['family', 'subfamily']

X = df.drop(columns=X_features)
y = df["critical_temp"]

#train & test data split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)
print(len(X_train),len(X_test))


import joblib
#scalling
scaler = RobustScaler()

#fit only on training data
X_train_scale = scaler.fit_transform(X_train)

#same scaling parameters for test data
X_test_scale = scaler.transform(X_test)
# ── Train XGBoost on scaled data ──────────────────────────
# Split a small validation set from training data only
X_tr, X_val, y_tr, y_val = train_test_split( X_train_scale, y_train, test_size=0.05, random_state=42)

model = xgb.XGBRegressor(
    n_estimators     = 1500,
    learning_rate    = 0.05,
    max_depth        = 11,
    subsample        = 0.75,
    colsample_bytree = 0.65,
    random_state     = 42,
    n_jobs           = -1,
    objective        = "reg:squarederror"
)
model.fit(
    X_tr, y_tr,
    eval_set = [(X_val, y_val)],  # validation only, no test leakage
    verbose  = False
)

# ── Evaluate ───────────────────────────────────────────────
y_pred = model.predict(X_test_scale)
print(f"Model R² score: {r2_score(y_test, y_pred):.5f}")


In [ ]:

# ── SHAP on scaled data ────────────────────────────────────
explainer = shap.TreeExplainer(model)

sample_idx = np.random.RandomState(42).choice(
    len(X_train_scale), size=min(2000, len(X_train_scale)), replace=False
)
samples_for_shap = pd.DataFrame(
    X_train_scale[sample_idx],
    columns = X_train.columns   # scaled + column names preserved
)
shap_value = explainer.shap_values(samples_for_shap)

# ── Feature importance ─────────────────────────────────────
shap_imp = pd.Series(
    np.abs(shap_value).mean(axis=0),
    index = X_train.columns     # use X_train.columns not X.columns
).sort_values(ascending=False)

top_50_features_83 = shap_imp.head(30).index

print("Top 15 features:\n", shap_imp.head(15))

In [ ]:

# ── Step 1: Filter top 50 from UNSCALED data ──────────────
X_top50_train = X_train[top_50_features_83]   # unscaled DataFrame 
X_top50_test  = X_test[top_50_features_83]    # unscaled DataFrame 

# ── Step 2: Save as separate files if needed ──────────────
top50_train_83 = pd.concat([
    X_top50_train.reset_index(drop=True),
    y_train.reset_index(drop=True)
], axis=1)

top50_test_83 = pd.concat([
    X_top50_test.reset_index(drop=True),
    y_test.reset_index(drop=True)
], axis=1)

top50_train_83.to_csv('top30_vF_xi_Hc_train_83.csv', index=False)  # save 
top50_test_83.to_csv('top30_vF_xi_Hc_test_83.csv',   index=False)  # save 

print(top50_train_83.shape)  # (17010, 51)
print(top50_test_83.shape)   # (4253, 51)



In [ ]:
# ── Plot 1: Bar chart ──────────────────────────────────
shap.summary_plot(
    shap_value, samples_for_shap,
    plot_type    = "bar",
    max_display  = 15,
    show         = False
)
plt.xlabel('Average of absolute SHAP value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=600, bbox_inches='tight')
plt.show()


# ── Plot 2: Beeswarm ───────────────────────────────────
shap.summary_plot(
    shap_value, samples_for_shap,
    max_display = 15,
    show        = False,
    cmap='coolwarm' # Red/Blue color scheme
)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=600, bbox_inches='tight')
plt.show()
